# Практикум 13. Иерархическая кластеризация и DBSCAN

Сопровождает интерактивную лабораторию №13 («Иерархическая и DBSCAN»). Тема курса: агломеративная кластеризация и плотностная кластеризация

Ноутбук самодостаточен: первая ячейка создаёт файл `common.py` с учебным корпусом и вспомогательными функциями. Запускайте ячейки сверху вниз. Корпус учебный (шаблонный), для реальной задачи подставьте свой CSV со столбцами `text` и `label`.

In [ ]:
%%writefile common.py
"""Общие функции для практикумов по классификации и кластеризации текстов."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "спорт": ("матч команда гол игрок тренер чемпионат турнир победа поражение сезон стадион болельщик судья счёт "
              "защитник нападающий вратарь кубок лига финал трансфер клуб удар пенальти соревнование спортсмен "
              "рекорд медаль олимпиада тренировка ничья").split(),
    "политика": ("закон депутат выборы партия голосование президент министр правительство парламент реформа бюджет "
                 "санкции кандидат коалиция министерство указ поправка законопроект оппозиция референдум дипломатия "
                 "мэр губернатор конституция заседание комиссия переговоры саммит избиратель мандат полномочия").split(),
    "кулинария": ("суп рецепт соль мясо готовить духовка овощи сковорода кастрюля тесто сахар специи варить жарить "
                  "запекать соус бульон салат повар блюдо ингредиент масло мука яйцо молоко чеснок лук перец "
                  "картофель десерт выпечка начинка маринад").split(),
    "технологии": ("процессор программа приложение смартфон нейросеть алгоритм данные сервер обновление разработчик "
                   "компьютер память экран камера браузер сеть интернет облако код платформа пользователь устройство "
                   "чип батарея ноутбук софт видеокарта система интерфейс шифрование").split(),
}
NEUTRAL = ("сегодня очень новый говорят после который также можно было стал сообщили вчера недавно ещё всё "
           "несколько важный первый").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """Учебный корпус из коротких 'документов' четырёх тем.
    noise: доля слов, взятых не из своей темы (нейтральные или чужие). Чем больше noise, тем труднее задача."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """Проекция разреженной матрицы TF-IDF на плоскость (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Печатает слова с наибольшими весами для каждого класса (линейные модели)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))

In [ ]:
from common import *

# Учебный корпус: 4 темы. noise - доля "посторонних" слов (чем больше, тем труднее).
df = make_corpus(n_per_class=100, noise=0.3, seed=0)
# Свой корпус вместо учебного: df = pd.read_csv("texts.csv")  # столбцы: text, label
print(df.label.value_counts().to_dict())
df.head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline
from sklearn.cluster import AgglomerativeClustering, DBSCAN
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors

X = TfidfVectorizer().fit_transform(df.text); y = df.label.values
Xr = make_pipeline(TruncatedSVD(10, random_state=0), Normalizer(copy=False)).fit_transform(X)   # плотные нормированные векторы (10 измерений)
print(Xr.shape)

## 1. Агломеративная кластеризация: сравнение связей

In [ ]:
rows = []
for link in ["single", "complete", "average", "ward"]:
    lab = AgglomerativeClustering(4, linkage=link).fit_predict(Xr)
    rows.append((link, round(adjusted_rand_score(y, lab), 3)))
pd.DataFrame(rows, columns=["linkage", "ARI"])

## 2. Дендрограмма (подвыборка 60 документов)

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
idx = np.random.default_rng(0).choice(len(y), 60, replace=False)
Lk = linkage(Xr[idx], method="ward")
plt.figure(figsize=(13, 4)); dendrogram(Lk, labels=y[idx], leaf_font_size=8); plt.title("Дендрограмма (Ward)"); plt.show()

## 3. DBSCAN: подбор ε по графику расстояний

In [ ]:
k = 5
dist, _ = NearestNeighbors(n_neighbors=k).fit(Xr).kneighbors(Xr)
plt.plot(np.sort(dist[:, -1])[::-1]); plt.ylabel(f"расстояние до {k}-го соседа"); plt.xlabel("документы по убыванию"); plt.title("k-distance: ищем излом"); plt.show()

In [ ]:
rows = []
for eps in [0.3, 0.4, 0.5, 0.55, 0.6]:
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Xr)
    nc = len(set(lab)) - (1 if -1 in lab else 0); noise = (lab == -1).mean()
    ok = lab != -1
    rows.append((eps, nc, round(noise, 2), round(adjusted_rand_score(y[ok], lab[ok]), 3) if ok.sum() > 1 and nc > 1 else np.nan))
pd.DataFrame(rows, columns=["eps", "кластеров", "доля шума", "ARI (без шума)"])

## Задания

1. Подберите `eps` и `min_samples`, при которых DBSCAN находит ровно 4 кластера.
2. Сравните агломеративную кластеризацию с метрикой `cosine` и `linkage="average"` (параметр `metric` вместо `affinity`).
3. Запустите DBSCAN на полной матрице TF-IDF (`metric="cosine"`) без снижения размерности. Почему подобрать ε гораздо труднее?